# Exercise 11

This exercise is based on Chapter 8 (Point Pattern Analysis) of the Geographic Data Science book.

The material can be found in: `GSP538/gds_book/notebooks/08_point_pattern_analysis.ipynb`

#### Notes on Textbook

- There are a lot of theory and coding examples in this chapter. A deep understanding of this material is important. Give yourself extra time to study this material and complete this exercise.
- The authors mention the chi-squared test, but do not really define it.
  - This is a common test seen in statistics that can be used for multiple purposes. 
  - In this case, we use the chi-squared test to see if the number of points in each cell follows a uniform distribution. If the points were distributed uniformly, then we would expect to find approximately the same number of points in each cell. Rejecting the null hypothesis therefore means there is some irregular distribution of the points between the cells; oftentimes we attribute this to "clustering" but as the book shows, sometimes it can be due to an irregularly shaped study area. 
  - Here is a brief explanation of this with a worked out example. https://medium.com/@chirag1162/uniform-distribution-using-chi-squared-test-for-goodness-of-fit-in-r-and-hand-calculation-6628f6a12e84

#### Answer the following written questions

There is a blank Markdown cell after each question for your answer (double click in the blank cell to type your answer). Be sure to run your Markdown cells to format your answers.

1. Points can be considered as "fixed" or "site of measurement." Explain in your own words what this means and its importance. Give an example of each not seen in the book.

2. In your own words, what is meant by "marked" versus "unmarked" point patterns? Give an example of each not seen in the book.

3. In your own words, what is meant by "process" versus "pattern"? Give an example (not seen in the book) that demonstrates how process relates to pattern.

4. Explain in your own words how the values in a 2D KDE map are computed from point data.

5. What does it mean when the mean and median of a numeric distribution are not close together? What does it mean when the mean center and median center of a spatial distribution are not close together?

6. What is similar about convex hulls and alpha shapes? What is different?

7. Describe the concept of a completely spatially random (CSR) process. How does the bounding area affect simulating this process? (Hint: review the first two maps under the "Randomness & clustering" heading.)

8. Quadrat analysis is based on the chi-squared test. See the notes and linked reading at the top of this exercise for a brief explanation of the chi-squared test. Explain the null hypothesis of the quadrat test and how that links to the null hypothesis of the chi-squared test. (Hint: read the "Quadrat analysis section closely before answering this question.)

9. How are distances calculated for Ripley's G function versus Ripley's F function?

10. Explain the difference between identifying "clustering" versus "clusters" as described in this chapter.

#### The following questions require you to run Python code.

The cell below imports the GeoPandas package and the standard extra stuff to make things run smoother. Run this cell. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd

The cell below contains a function to match the Ripley plots demonstrated in the book. You will use this later in the exercise. Run this cell.

In [ ]:
def plot_ripley(test_results, coords):
    '''
    test_results: output of distance_statistics.g_test(), .f_test(), etc.
    
    coords: same list of coordinates passed to distance_statistics.g_test(), .f_test(), etc.
    '''
    
    f, ax = plt.subplots(1, 2, figsize=(9, 3), 
                         gridspec_kw=dict(width_ratios=(6, 3)))

    # plot all the simulations with very fine lines
    ax[0].plot(test_results.support, test_results.simulations.T, 
               color="k", alpha=0.01)
    
    # and show the average of simulations
    ax[0].plot(test_results.support, np.median(test_results.simulations, axis=0),
        color="cyan", label="median simulation")

    # and the observed pattern's Ripley function
    ax[0].plot(test_results.support, test_results.statistic, label="observed", 
               color="red")

    # clean up labels and axes
    ax[0].set_xlabel("distance (m)")
    ax[0].set_ylabel("% of nearest neighbor\ndistances shorter")
    ax[0].legend()
    ax[0].set_xlim(0, 40000)
    letter = str(type(test_results)).split('.')[2][0]
    ax[0].set_title(r"Ripley's $" + letter + "(d)$ function")

    # plot the pattern itself on the next frame
    ax[1].scatter(*coords.T)

    # and clean up labels and axes there, too
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticklabels([])
    ax[1].set_yticklabels([])
    ax[1].set_title("Pattern")
    f.tight_layout()
    plt.show()

11. The cell below reads in the fires dataset, subsets it to Arizona and converts it to a GeoDataFrame. Run the cell.
    - Describe what the last three lines are doing
    - Why, for the analyses in this chapter, should you use these values instead of latitude/longitude?

In [ ]:
# Read in fires data
fires = pd.read_csv('data/fpa_az_ca_fires.csv')
# Convert to GeoDataFrame
pt_geoms = gpd.points_from_xy(x=fires["LONGITUDE"],
                              y=fires["LATITUDE"],
                              crs="EPSG:4326")
fires = gpd.GeoDataFrame(fires, geometry=pt_geoms)
# Subset to Arizona
fires = fires.loc[fires.STATE=='AZ',]

fires = fires.to_crs(32612)          # L1
fires['x_coord'] = fires.geometry.x  # L2
fires['y_coord'] = fires.geometry.y  # L3 

12. Use Seaborn's `jointplot` to plot the points over a contextily basemap showing terrain.
    - Color the points red so that they show up against the basemap
    - Make the points large enough so that they can be seen, but are not exaggerated
    - Make sure the map is in the projected coordinates used for the GeoDataFrame
    - What do the bars along the top and side of the map represent?
    - Interpret the results

13. Create a hexbin map of the fires data.
    - Include a contextily base map
    - Include a colorbar legend
    - Use the `viridis_r` colormap (or another colormap that might better demonstrate the results)
    - Use hexagons that are large enough to see the color clearly
    - Use transparency to see some of the basemap underneath
    - Turn off the coordinate axis
    - Make the figure larger
    - You are encouraged to use the Pandas hexbin plotting tool (i.e., `fires.plot.hexbin()`) instead of the Matplotlib version in the book. It is fewer lines of code.
    
    <br>
    
    Create a KDE map of the fires data.
    - Include a contextily base map
    - Use the `viridis_r` colormap (or another colormap that might better demonstrate the results)
    - Use transparency to see some of the basemap underneath
    - Turn off the coordinate axis
    - You are encouraged to use `seaborn.kdeplot()`
    - Make the figure larger (Hint: you can precede `seaborn.kdeplot()` with `plt.figure(figsize=(10,10))` to make it larger.)
    
    <br>
    
    Compare the maps and interpret the distribution of fires in Arizona.

14. Centography statistics. 
    - Compute the mean center for the fires dataset and print the results
    - Compute the median center for the fires dataset and print the results
    - Compute the standard distance for the fires dataset and print the results<br><br>
   
    Centography plot
    
    - Hint: build one map layer and test it to make sure it plots; then add the next map layer and test it; continue until you have all the layers plotted
    - Plot the fire points
      - Red points; not too big, not too small
      - You are encouraged to use the plot function of the GeoDataFrame (i.e., `fires.plot()`)
    - Plot the mean center as a blue "x"
    - Plot the median center as a large limegreen point
    - Plot the standard ellipse as a blue dotted line
    - Include a legend containing the symbology for mean, median and ellipse
    - Make the figure larger
    - Include a contextily basemap

15. The book discusses the concept of a "minimum bounding rectangle", which is simply a synonym for "bounding box" that you have seen in other courses. 
    - Create a new GeoDataFrame with just those fires over 50,000 acres
    - Create a second object with just the X and Y coordinates for these big fires as a Numpy array (Hint: the authors show how to do this in the book.)
    - Create the minimum bounding rectangle using the centography package and print the resulting object
    - Create a bounding box directly using the GeoDataFrame method `total_bounds` (e.g., `fires_big.total_bounds`) and print the resulting object
    - Discussion
      - Compare the two results. Are they the same or different? 
      - Give an explanation of each of the four values. (Hint: read the documentation for each tool: [GeoPandas](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.total_bounds.html) and 
    [PySAL](https://pysal.org/notebooks/explore/pointpats/centrography.html#Minimum-Bounding-Rectangle).)
      - Why do you only need these particular four values to fully describe the rectangle's coordinates? 
    - Note: Do not create a map here, you will do this in later question

16. In the previous question you implemented the most common way of measuring the _extent_ of a point pattern. In this question you will implement two more approaches.
    - Create the alpha shape for the big fires
      - What type of Python object is it?
      - Print out the coordinates of the object (Hint: something like `my_alpha_shape.exterior.coords[0:]`)
    - Create the convex hull for the big fires
      - What type of Python object is it?
      - Print out the coordinates of the object (Hint: you can just type the object name like `my_convex_hull`)
    - Note: Do not create a map here, you will do this in later question

17. Create a single map that shows all three measures of extent for the big fires. As you saw in the previous questions, each is a different type of object. Furthermore, the bounding box does not even use coordinate pairs like the other two do. This means that there are differences in how each is plotted. The good news is that the book demonstrates how to build this map.
    - Hint: build one map layer and test it to make sure it plots; then add the next map layer and test it; continue until you have all the layers plotted
    - Plot all fires (red points; not too big, not too small)
    - Plot the big fires (yellow points; bigger than the red points)
    - Plot the alpha shape for the big fires
    - Plot the convex hull for the big fires (different color and/or line style from the alpha shape)
    - Plot the bounding rectangle for the big fires (different color and/or line style from the alpha shape and convex hull)
      - Note: there is a typo in the book. The line `min_rect_height = min_rect_vertices[2] - min_rect_vertices[0]` should be `min_rect_height = min_rect_vertices[3] - min_rect_vertices[1]`
      - An (easier) alternative to the book's approach is to first convert the bounding rectangle object into a shapely polygon using the shapely function `box`. It can then be plotted like the other shapely polygon.
    - Increase the size of the map
    - Remove the coordinate axis
    - Include a legend containing the three measures of extent
    - Include a contextily basemap
    - Finally, interpret your map

18. For each of the following cases: 1) run the quadrat statistic, 2) print the p-value 3) print the quadrat grid map and 4) discuss the results including whether or not you can reject the null hypothesis.
    - Big fire points (using default 3x3 grid)
    - All fire points (using default 3x3 grid)
    - All fire points (using 8x10 grid) (Hint: [see the documentation](https://pysal.org/notebooks/explore/pointpats/Quadrat_statistics.html))

19. Ripley's tests.
    - Ripley's G for for fires started by Arson
      - Compute Ripley's G test (Note use: `support=40, n_simulations=999, keep_simulations=True`)
      - Plot the results using the `plot_ripley()` function that was provided earlier in this notebook 
    - Ripley's G for for fires started by Campfire
      - Compute Ripley's G test (Note use: `support=40, n_simulations=999, keep_simulations=True`)
      - Plot the results using the `plot_ripley()` function that was provided earlier in this notebook 
    - Interpret your results
      - Within the chart: the actual pattern versus the simulated pattern
      - Across the charts: the arson pattern is slightly different from the campfire pattern, discuss the difference

20. The following cell runs DBSCAN and outputs some results.
    - Describe each line of code in detail
    - Interpret the results

In [ ]:
from sklearn.cluster import DBSCAN  #L1

clusterer = DBSCAN(eps=10000, min_samples=20)  #L2
clusterer.fit(fires[['x_coord','y_coord']])    #L3

lbls = pd.Series(clusterer.labels_, index=fires.index, name='clusterID')  #L4
fires2 = pd.concat([fires,lbls], axis=1)  #L5

print(fires2.clusterID.value_counts())    #L6

fires2.loc[fires2.clusterID!=-1].explore(column='clusterID', categorical=True,
                                         tooltip=['MTBS_FIRE_NAME','FIRE_SIZE',
                                                  'STAT_CAUSE_DESCR','FIRE_YEAR',
                                                  'FIPS_NAME','clusterID'],
                                         legend=False)                            #L7